In [1]:
import pandas as pd
import sqlite3
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

In [5]:
# Load data from database

conn = sqlite3.connect('../data/customer_tickets.db')
df = pd.read_sql_query("SELECT * FROM support_tickets", conn)
print("Original shape:", df.shape)

df.head()

Original shape: (1820, 9)


,issue_key,issue_created,issue_summary,issue_description,issue_reporter,division,issue_priority,issue_assignee,issue_status
0,1,26-08-24,Programs for model-n simplication and eliminat...,Project for IMAS update bypassing Model-N,THOPPILR,ELX,MEDIUM,THOPPILR,READY FOR GO LIVE
1,2,07-03-25,Ship and Debit changes,,THOPPILR,,MEDIUM,THOPPILR,OPEN
2,3,13-11-23,Ship-to Level Credit Term,,LILE,ELX,MEDIUM,LILE,CLOSED
3,4,14-11-23,Migrate invoice format 02 & 06,,LILE,ELX,MEDIUM,LILE,CLOSED
4,5,14-11-23,combine multiple interco payment in bank file,,LILE,ELX,MEDIUM,LILE,CLOSED


In [7]:
# Drop rows with missing values in key columns

df = df.dropna(subset=['issue_priority', 'issue_summary', 'issue_description']).reset_index(drop=True)
print("After dropping nulls:", df.shape)
df.head()

After dropping nulls: (1820, 9)


,issue_key,issue_created,issue_summary,issue_description,issue_reporter,division,issue_priority,issue_assignee,issue_status
0,1,26-08-24,Programs for model-n simplication and eliminat...,Project for IMAS update bypassing Model-N,THOPPILR,ELX,MEDIUM,THOPPILR,READY FOR GO LIVE
1,2,07-03-25,Ship and Debit changes,,THOPPILR,,MEDIUM,THOPPILR,OPEN
2,3,13-11-23,Ship-to Level Credit Term,,LILE,ELX,MEDIUM,LILE,CLOSED
3,4,14-11-23,Migrate invoice format 02 & 06,,LILE,ELX,MEDIUM,LILE,CLOSED
4,5,14-11-23,combine multiple interco payment in bank file,,LILE,ELX,MEDIUM,LILE,CLOSED


In [9]:
# Map priorities to 3-class labels

priority_mapping = {
    'BLOCKER': 'High',
    'HIGH': 'High',
    'MEDIUM': 'Medium',
    'LOW': 'Low',
    'VERY LOW': 'Low'
}

df['priority_class'] = df['issue_priority'].map(priority_mapping)
df = df.dropna(subset=['priority_class'])  # remove rows that didn’t map
df['priority_class'].value_counts()

priority_class
Medium    1558
High       213
Low         49
Name: count, dtype: int64

In [10]:
# Text cleaning
stop_words = set([
    'the', 'is', 'in', 'and', 'to', 'with', 'a', 'of', 'for', 'on', 'at', 'by', 'an', 'be'
])

def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return ' '.join(word for word in text.split() if word not in stop_words)

df['text'] = df['issue_summary'].fillna('') + ' ' + df['issue_description'].fillna('')
df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text']].head()

,text,clean_text
0,Programs for model-n simplication and eliminat...,programs modeln simplication elimination imas ...
1,Ship and Debit changes,ship debit changes
2,Ship-to Level Credit Term,shipto level credit term
3,Migrate invoice format 02 & 06,migrate invoice format
4,combine multiple interco payment in bank file,combine multiple interco payment bank file


In [11]:
# Label Encoding
le = LabelEncoder()
df['priority_encoded'] = le.fit_transform(df['priority_class'])
print("Class Mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

Class Mapping: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}


In [ ]:
# Save cleaned dataset

df.to_csv('../data/support_tickets_cleaned.csv', index=False)
print("Cleaned dataset saved.")

Cleaned dataset saved.
